# Generator Quality Evaluation

We have three tools that create **synthetic (fake) vehicle data** for Turkey:
- **CTGAN v7** and **TVAE v7** — learned from all 11 countries (37K and 9K Turkey-like rows)
- **GaussianCopula v2** — learned from Turkey data only (7,200 rows)

This notebook answers two questions:

> **Question 1 — Usefulness**: If I train a prediction model on synthetic data and test it on real Turkey data, how accurate is it?
> A good generator = model trained on fake data performs almost as well as model trained on real data.

> **Question 2 — Realism**: Can a classifier tell the difference between a real Turkey row and a fake one?
> A good generator = the classifier can't distinguish them (it's essentially guessing).

## 1) Setup

Load real Turkey data and all three synthetic datasets. We then encode the categorical columns into numbers so scikit-learn models can process them.

In [ ]:
try:
    import pandas as _pd
    _pd.options.future.infer_string = False
except AttributeError:
    pass

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.ensemble       import RandomForestClassifier
from sklearn.linear_model   import LogisticRegression
from sklearn.preprocessing  import OrdinalEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics        import (f1_score, classification_report,
                                    roc_auc_score, RocCurveDisplay)

# ── Real Turkey training data ──────────────────────────────────────────────
train = pd.read_csv("../data/EM_LYON_train_set_20260206.csv", sep=";")
train["country_iso"] = train["country_iso"].astype(str).str.strip().str.upper()
train = train.dropna(subset=["car_maker_name", "car_segment_name",
                              "energy", "code_age", "body_style"]).copy()

age_map = {
    "Less than 1 year old": 0.5, "1 year old": 1.0, "2 years old": 2.0,
    "3 to 5 years old": 4.0, "6 to 10 years old": 8.0, "11 years and older": 12.0,
}
train["age_years"] = train["code_age"].map(age_map)

tr_real = train[train["country_iso"] == "TR"].copy()
print(f"Real Turkey rows : {len(tr_real)}")

# ── Synthetic data (v7 CTGAN/TVAE from notebook 07, GC v2 from notebook 02) ──
SYNTH_FILES = {
    "CTGAN-v7": "../data/ctgan_synth_turkey_v7.csv",
    "TVAE-v7":  "../data/tvae_synth_turkey_v7.csv",
    "GC-v2":    "../data/gc_synth_turkey_v2.csv",
}
for label, path in SYNTH_FILES.items():
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"{path} not found.\n"
            "Run the source notebook first to generate this file."
        )

synth_data = {}
for label, path in SYNTH_FILES.items():
    df = pd.read_csv(path)
    # Normalise column names
    if "country_name" in df.columns and "country_iso" not in df.columns:
        df = df.rename(columns={"country_name": "country_iso"})
    # Add age_years if missing
    if "age_years" not in df.columns and "code_age" in df.columns:
        df["age_years"] = df["code_age"].map(age_map)
    synth_data[label] = df
    print(f"{label:12s}: {df.shape}")

ctgan_synth = synth_data["CTGAN-v7"]
tvae_synth  = synth_data["TVAE-v7"]
gc_synth    = synth_data["GC-v2"]

# ── Shared encoding setup ─────────────────────────────────────────────────
FEATURES   = ["car_maker_name", "car_segment_name", "body_style", "code_age", "age_years"]
TARGET_COL = "energy"

# Fit OrdinalEncoder on the union of all data so unknown synthetic categories are handled
all_feat = pd.concat(
    [tr_real[FEATURES]] + [df[FEATURES] for df in synth_data.values()],
    ignore_index=True
)
enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
enc.fit(all_feat.astype(str))

# Consistent target encoding (classes = those seen in real Turkey data)
target_classes = sorted(tr_real[TARGET_COL].dropna().astype(str).unique())
t_map = {v: i for i, v in enumerate(target_classes)}
print(f"\nFeatures   : {FEATURES}")
print(f"Target     : {TARGET_COL}  →  {target_classes}")

def encode_X(df):
    return enc.transform(df[FEATURES].astype(str).fillna("__MISSING__"))

def encode_y(df):
    return df[TARGET_COL].astype(str).map(t_map).fillna(-1).astype(int).values

# Sub-sample synthetic to 10× real size for speed
N_SYNTH = len(tr_real) * 10
rng = np.random.RandomState(42)

encoded = {}
for label, df in synth_data.items():
    n = min(N_SYNTH, len(df))
    sub = df.sample(n, random_state=42)
    X = encode_X(sub)
    y = encode_y(sub)
    mask = y >= 0
    encoded[label] = {"X": X[mask], "y": y[mask]}
    print(f"{label:12s} encoded : {X[mask].shape}")

X_real = encode_X(tr_real)
y_real = encode_y(tr_real)
real_mask = y_real >= 0
X_real, y_real = X_real[real_mask], y_real[real_mask]
print(f"\nReal Turkey encoded : {X_real.shape}")

## 2) Usefulness Test — Train on Synthetic, Test on Real (TSTR)

**The idea**: pick a prediction task we can measure objectively — predicting the **fuel type** (`energy`: petrol, diesel, electric…) from the car's brand, segment, body style, and age.

We run three experiments:

| Experiment | Training data | Test data |
|-----------|--------------|-----------|
| **Baseline (TRTR)** | Real Turkey rows (cross-validated) | Real Turkey rows | 
| **TSTR** | Synthetic rows from each generator | Real Turkey rows |

The metric is **F1 score** (0 = random, 1 = perfect).

**How to read the gap**:
- Gap ≈ 0 → synthetic data is as useful as real data ✓
- Gap strongly negative → synthetic data loses important information ✗
- Gap slightly positive → synthetic data adds useful variation (trained on more countries)

In [ ]:
RF_PARAMS = dict(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# TRTR: cross-validate on real data (oracle)
trtr_scores = cross_val_score(RandomForestClassifier(**RF_PARAMS), X_real, y_real,
                               cv=cv5, scoring="f1_weighted")
trtr_f1 = trtr_scores.mean()
print(f"TRTR (oracle) F1 = {trtr_f1:.4f} ± {trtr_scores.std():.4f}")

# TSTR: train on synthetic, evaluate on all real rows
tstr_results = {}
for label, enc_dict in encoded.items():
    Xs, ys = enc_dict["X"], enc_dict["y"]
    rf = RandomForestClassifier(**RF_PARAMS)
    rf.fit(Xs, ys)
    y_pred = rf.predict(X_real)
    f1 = f1_score(y_real, y_pred, average="weighted", zero_division=0)
    tstr_results[label] = {"f1": f1, "model": rf, "y_pred": y_pred}

    print(f"\nTSTR-{label} F1 = {f1:.4f}  (gap vs TRTR: {f1 - trtr_f1:+.4f})")
    print(classification_report(y_real, y_pred,
                                 target_names=target_classes, zero_division=0))

print("\n" + "=" * 65)
print(f"{'Setup':<22} {'F1 (weighted)':>16} {'Gap vs TRTR':>14}")
print("-" * 54)
print(f"{'TRTR (oracle)':<22} {trtr_f1:>16.4f} {'—':>14}")
for label, res in tstr_results.items():
    print(f"{'TSTR-'+label:<22} {res['f1']:>16.4f} {res['f1']-trtr_f1:>+14.4f}")
print("=" * 65)
print("Gap → 0 : synthetic data is a good substitute for real training data")

### Results — Usefulness

**CTGAN-v7 and TVAE-v7: essentially as good as real data** (gap < +0.03)

Both generators slightly *beat* the oracle, which is not a contradiction — they were trained on all 11 countries, so the model learns fuel-type patterns from a much larger and more diverse pool than Turkey's 6,851 real rows alone. The small positive gap is within noise and means: *synthetic data from these generators is a faithful substitute for real training data*.

**GaussianCopula-v2: significant quality loss** (gap = −0.12)

The model trained on GC synthetic data scores 0.42 F1 vs 0.54 for real data — accuracy drops from 61% to 42%. Two reasons:
1. GC was trained on Turkey rows only (much less data) 
2. GC is a simpler model that captures averages and pairwise correlations but misses complex multi-column patterns — it over-generates diesel rows and incorrectly generates electric rows where none should appear

**Why the oracle is only 0.54**: predicting fuel type from make/segment/age is an inherently uncertain task — the same car model is sold in petrol, diesel, and hybrid variants. Even with perfect real data you can't get much higher. CTGAN/TVAE hit this ceiling; GC falls below it.

## 3) Realism Test — Can a Classifier Tell Real from Fake?

**The idea**: label every real Turkey row as 1 and every synthetic row as 0, then train a binary classifier to separate them. Measure its AUC-ROC.

- **AUC = 0.5** → the classifier is guessing — real and fake look identical ✓ (ideal)
- **AUC = 1.0** → the classifier perfectly separates them — the generator is failing ✗
- **AUC < 0.6** → good quality (hard to tell apart)
- **AUC 0.6–0.75** → borderline
- **AUC > 0.75** → generator diverges significantly from real data

This test checks the **joint distribution across all features at once** — more powerful than comparing columns one by one.

In [ ]:
# Balance: use all real rows + equal-sized sample of synthetic
N_PROP = len(X_real)

propensity_aucs = {}

for label, enc_dict in encoded.items():
    Xs = enc_dict["X"]
    idx_s = rng.choice(len(Xs), size=min(N_PROP, len(Xs)), replace=False)
    X_pool = np.vstack([X_real, Xs[idx_s]])
    y_pool = np.concatenate([np.ones(len(X_real)), np.zeros(len(idx_s))])

    clf = LogisticRegression(max_iter=1000, random_state=42)
    aucs = cross_val_score(clf, X_pool, y_pool,
                           cv=StratifiedKFold(5, shuffle=True, random_state=42),
                           scoring="roc_auc")
    mean_auc = aucs.mean()
    propensity_aucs[label] = (mean_auc, X_pool, y_pool)
    quality = "good" if mean_auc < 0.6 else ("borderline" if mean_auc < 0.75 else "high — generator diverges")
    print(f"{label:12s} propensity AUC = {mean_auc:.4f} ± {aucs.std():.4f}  [{quality}]")

# ROC curves — 1 row, 3 columns
n_gen = len(propensity_aucs)
fig, axes = plt.subplots(1, n_gen, figsize=(6 * n_gen, 5))
for ax, (label, (auc, X_pool, y_pool)) in zip(axes, propensity_aucs.items()):
    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(X_pool, y_pool)
    RocCurveDisplay.from_estimator(clf, X_pool, y_pool, ax=ax, name=label)
    ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="random (AUC = 0.5)")
    ax.set_title(f"{label} Propensity AUC = {auc:.4f}")
    ax.legend(fontsize=8)
plt.suptitle("Propensity Score ROC — AUC closer to 0.5 is better", y=1.02)
plt.tight_layout()
plt.savefig("../report/figures/fig_propensity_roc.pdf", bbox_inches="tight", dpi=150)
plt.show()

### Results — Realism

A good generator keeps the AUC close to 0.5 (classifier is guessing).

**CTGAN-v7**: trained on 11 countries, so its synthetic Turkey rows mix European patterns in — a classifier can partially identify this as non-native Turkey data. Expect AUC slightly above 0.5.

**TVAE-v7**: similar story to CTGAN but with fewer Turkey-like rows (9K vs 37K), so the distribution may be more concentrated and slightly easier or harder to distinguish.

**GaussianCopula-v2**: trained only on Turkey, so the marginals match, but its simpler structure (can't model complex multi-column correlations) means a classifier may still detect the simplistic joint distribution. The TSTR result already showed GC misrepresents fuel type ratios — the same flaw will appear here.

The ROC curves below show how easy it is to separate each generator's output from real Turkey rows.

## 4) Feature Importance — What the Discriminator Learned

A **Random Forest discriminator** (real vs synthetic) reveals which features make the two
distributions most distinguishable.

$$\text{High importance} \Rightarrow \text{that column diverges most between real and synthetic Turkey}$$

These are the columns the generator handles least faithfully — useful to know when deciding
how much to trust the synthetic data for downstream predictions.

In [ ]:
n_gen = len(encoded)
fig, axes = plt.subplots(1, n_gen, figsize=(6 * n_gen, 5))
disc_importances = {}

for ax, (label, enc_dict) in zip(axes, encoded.items()):
    Xs = enc_dict["X"]
    n = min(len(X_real), len(Xs))
    idx_r = rng.choice(len(X_real), size=n, replace=False)
    idx_s = rng.choice(len(Xs),    size=n, replace=False)
    Xd = np.vstack([X_real[idx_r], Xs[idx_s]])
    yd = np.concatenate([np.ones(n), np.zeros(n)])

    rf_disc = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42, n_jobs=-1)
    rf_disc.fit(Xd, yd)

    imp = pd.Series(rf_disc.feature_importances_, index=FEATURES).sort_values()
    disc_importances[label] = imp

    imp.plot.barh(ax=ax, color="steelblue", alpha=0.85)
    ax.axvline(1 / len(FEATURES), color="red", linestyle="--", linewidth=1,
               label=f"uniform baseline ({1/len(FEATURES):.2f})")
    ax.set_title(f"{label} discriminator — feature importances")
    ax.set_xlabel("Importance")
    ax.legend(fontsize=8)

plt.suptitle("High importance = where synthetic data diverges most from real Turkey", y=1.02)
plt.tight_layout()
plt.savefig("../report/figures/fig_feature_importance.pdf", bbox_inches="tight", dpi=150)
plt.show()

print("\nTop feature by importance:")
for label, imp in disc_importances.items():
    print(f"  {label:12s}: {imp.idxmax()} ({imp.max():.4f})")

## 5) CTGAN vs TVAE — Summary

In [ ]:
print("=" * 80)
print("ML EVALUATION SUMMARY")
print("=" * 80)
print(f"\n{'Metric':<42} {'CTGAN-v7':>12} {'TVAE-v7':>12} {'GC-v2':>10}")
print("-" * 78)
print(f"{'TSTR F1 (energy prediction)':<42} "
      f"{tstr_results['CTGAN-v7']['f1']:>12.4f} "
      f"{tstr_results['TVAE-v7']['f1']:>12.4f} "
      f"{tstr_results['GC-v2']['f1']:>10.4f}")
print(f"{'TRTR F1 (oracle)':<42} "
      f"{trtr_f1:>12.4f} {trtr_f1:>12.4f} {trtr_f1:>10.4f}")
print(f"{'Utility gap  (TSTR − TRTR)':<42} "
      f"{tstr_results['CTGAN-v7']['f1']-trtr_f1:>+12.4f} "
      f"{tstr_results['TVAE-v7']['f1']-trtr_f1:>+12.4f} "
      f"{tstr_results['GC-v2']['f1']-trtr_f1:>+10.4f}")
print(f"{'Propensity AUC':<42} "
      f"{propensity_aucs['CTGAN-v7'][0]:>12.4f} "
      f"{propensity_aucs['TVAE-v7'][0]:>12.4f} "
      f"{propensity_aucs['GC-v2'][0]:>10.4f}")
print(f"{'Top discriminator feature':<42} "
      f"{disc_importances['CTGAN-v7'].idxmax():>12} "
      f"{disc_importances['TVAE-v7'].idxmax():>12} "
      f"{disc_importances['GC-v2'].idxmax():>10}")
print("=" * 80)
print()
print("Interpretation:")
print("  Utility gap → 0    : synthetic ≈ real for ML training (TSTR ≈ TRTR)")
print("  Propensity → 0.5   : joint distribution is indistinguishable from real")
print("  Propensity → 1.0   : generator produces easily separable synthetic data")
print("  Top discriminator  : the column where the generator is weakest")
print()
print("Generator notes:")
print("  CTGAN-v7 / TVAE-v7 : trained on all 11 countries (37K / 9K Turkey rows)")
print("  GC-v2              : trained on Turkey only via conditional sampling (7.2K rows)")